# Evolutionary Algorithm - Real-World Semiconductor Workshop Scheduling

In this notebook we will solve a real world scheduling problem using population based **Evolutionary Algorithm**.

## Data Structure

Our data structure is organised like this: ...

In [1]:
import json


class InstanceData:
    """InstanceData is a class that loads the instance data from a JSON file and provides access to the data."""

    def __init__(self, json_path: str = '../examples/75_3_5_H.json'):
        self.json_data = {}
        with open(json_path, 'r') as f:
            self.json_data = json.load(f)

    def __str__(self):
        return self.json_data.__str__()

instance = InstanceData()
print(f'Number of Jobs: {instance.json_data["n"]}')
print(f'Number of Machines: {instance.json_data["m"]}')
print(f'Horizon: {instance.json_data["horizon"]}')
print(f'Capable: {instance.json_data["capable"]}')
print(f'Duration: {instance.json_data["duration"]}')
print(f'Release: {instance.json_data["release"]}')
print(f'Setup: {instance.json_data["setup"]}')

Number of Jobs: 5
Number of Machines: 3
Horizon: 1664
Capable: [[2], [2], [2], [2], [2, 0, 1]]
Duration: [[175, 465, 352], [365, 452, 244], [193, 198, 156], [471, 281, 87], [57, 62, 59]]
Release: [[254, 122, 76], [67, 36, 83], [288, 23, 138], [141, 231, 341], [202, 20, 170]]
Setup: [[[0, 0, 0], [41, 93, 3], [32, 75, 67], [59, 56, 70], [61, 91, 78]], [[74, 3, 14], [0, 0, 0], [5, 7, 55], [4, 71, 83], [76, 94, 82]], [[37, 71, 2], [88, 13, 8], [0, 0, 0], [100, 97, 93], [10, 29, 72]], [[48, 84, 81], [95, 24, 85], [1, 27, 84], [0, 0, 0], [51, 77, 41]], [[59, 29, 10], [66, 28, 20], [85, 74, 49], [72, 79, 18], [0, 0, 0]]]


The Data structure used to represent the population is: `List[List[int]]`, a list with each index a machine and each value a list of job id.

In [2]:
from typing import List

class Individual:
    def __init__(self, scheduling: List[List[int]]):
        self.scheduling = scheduling

    # def __str__(self):
    #     return self.scheduling.__str__()

    # def __repr__(self):
    #     return self.scheduling.__repr__()

# Example:
individual: Individual = Individual([
    [0, 1, 2],  # Machine 0 has jobs 0, 1, and 2
    [3, 4],     # Machine 1 has jobs 3 and 4
    [5]         # Machine 2 has job 5
])
print(individual.scheduling)

[[0, 1, 2], [3, 4], [5]]


## Algorithm

We have our main Evolutionary Algorithm there.
Our main component:
* ...
* ...

In [3]:
import random
from typing import Dict, List, Set, Tuple

import numpy as np

class GeneticAlgorithm:
    def __init__(self, instance: InstanceData, seed: int | None = None):
        self.instance = instance
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def add_one_to_job_ids(job_id: List[int]) -> List[int]:
        return [job + 1 for job in job_id]

    def format_solution(self, ind: Individual, makespan: float) -> Dict:
        return {
            "makespan": makespan,
            "schedule": {
                str(machine_id): GeneticAlgorithm.add_one_to_job_ids(machine_jobs) for machine_id, machine_jobs in enumerate(ind.scheduling)
            },
        }

    def init_population(
        self,
        pop_size: int = 50,
    ) -> List[Individual]:
        """
        Initialize a population of size pop_size.
        """
        population = []
        for _ in range(pop_size):
            # Generating each machine scheduling as empty
            scheduling = [[] for _ in range(self.instance.json_data['m'])]
            for job_id in range(self.instance.json_data['n']):
                # Placing a job where he can be done
                scheduling[random.sample(self.instance.json_data['capable'][job_id], 1)[0]].append(job_id)
            population.append(Individual(scheduling))
        return population

    def evaluate_individual(self, individual: Individual) -> float:
        """
        Returns the makespan of the schedule represented by the individual.
        """
        actual_time = 0.0

        for machine_id, machine_jobs in enumerate(individual.scheduling):
            machine_time = 0.0
            for i, job_id in enumerate(machine_jobs):
                job_release = self.instance.json_data['release'][job_id][machine_id]
                job_duration = self.instance.json_data['duration'][job_id][machine_id]
                job_setup = self.instance.json_data['setup'][machine_jobs[i - 1]][job_id][machine_id] if i > 0 else 0

                # The current machine time is the max betwee the earliest time the job can start
                # AND the time the machine is available after the previous job (if case is applicable)
                # Then we add the job duration to get the time of the machine after the job.
                machine_time = max(job_release, machine_time + job_setup) + job_duration

            actual_time = max(actual_time, machine_time)
        return actual_time


    def evaluate_population(self, population: List[Individual]) -> List[float]:
        """
        Evaluate each individual of the population.
        """
        makespan_scores = []
        for individual in population:
            makespan = self.evaluate_individual(individual)
            makespan_scores.append(makespan)
        return makespan_scores


    def crossover_operator(self, parent1: Individual, parent2: Individual) -> Tuple[Individual, Individual]:
        """
        Crossover operator that takes two parents and produces a child.
        We use a simple one-point crossover for each machine.
        """
        nb_machines = self.instance.json_data['m']

        # Create the 2 offsprings with empty scheduling
        offspring_1: Individual = Individual([[] for _ in range(nb_machines)])
        offspring_2: Individual = Individual([[] for _ in range(nb_machines)])

        # Sets of whats jobs are already assigned to the offsprings
        assigned_offspring_1: Set[int] = set()
        assigned_offspring_2: Set[int] = set()

        for machine_id in range(nb_machines):
            # Take a point for parent 1
            point = random.randint(0, len(parent1.scheduling[machine_id]))

            # All the jobs before the point are assigned to offspring 1, the others to offspring 2
            for job_id in parent1.scheduling[machine_id][:point]:
                offspring_1.scheduling[machine_id].append(job_id)
                assigned_offspring_1.add(job_id)
            for job_id in parent1.scheduling[machine_id][point:]:
                offspring_2.scheduling[machine_id].append(job_id)
                assigned_offspring_2.add(job_id)

        for machine_id in range(nb_machines):
            # All the jobs of parent 2 not in offspring 1 are assigned to it, the others to offspring 2
            for job_id in parent2.scheduling[machine_id]:
                if job_id not in assigned_offspring_1:
                    offspring_1.scheduling[machine_id].append(job_id)
                    assigned_offspring_1.add(job_id)
                if job_id not in assigned_offspring_2:
                    offspring_2.scheduling[machine_id].append(job_id)
                    assigned_offspring_2.add(job_id)
        return offspring_1, offspring_2

    def evolutionary_algorithm(
        self,
        pop_size: int,
        nb_generation: int = 50,
        mu: int = 150,
        lamda: int = 100
    ):
        if pop_size <= 0 or nb_generation <= 0:
            raise ValueError("Population size and number of generations must be positive integers.")

        # Init the population
        pops: List[Individual] = self.init_population(pop_size)

        # The number of generations we want to run the algorithm for and its intial value saved
        generations = nb_generation
        initial_generations = nb_generation

        # We store a progress history to plot the progress of the algorithm
        progress_history = []
        while generations > 0:
            # Evaluate the population
            makespans = self.evaluate_population(pops)


            # The list of offsprings generated by the crossover operator
            offsprings: List[Individual] = []

            j = 0
            while j < lamda:
                parents_chosen_idx = random.choices(range(len(pops)), k=2)
                parents_chosen = [pops[idx] for idx in parents_chosen_idx]

                child1, child2 = self.crossover_operator(parents_chosen[0], parents_chosen[1])

                offsprings.append(child1)
                offsprings.append(child2)
                j += 2

            # We use the strategy (mu + lambda) to select the best individuals for the next generation
            pops: List[Individual] = pops + offsprings
            pops = sorted(pops, key=lambda ind: self.evaluate_individual(ind))[:mu]
            makespans = [self.evaluate_individual(individual) for individual in pops]

            generations -= 1
            current_generation = initial_generations - generations

            best_current = min(makespans)
            progress_history.append((current_generation, best_current))
            print(f"Generation {current_generation} - Best Makespan: {best_current}")

        # Get the best offspring
        best_solution_idx = np.argmin(makespans)
        best_solution = pops[best_solution_idx]
        return self.format_solution(best_solution, makespans[best_solution_idx]), progress_history

In [6]:
import sys
sys.path.append('..')
from checker import check_and_evaluate

evolutionary_algo = GeneticAlgorithm(instance)
best_solution, progress_history = evolutionary_algo.evolutionary_algorithm(pop_size=50, nb_generation=50, mu=150, lamda=100)

print(f"Best solution found: {best_solution}")
print(check_and_evaluate(instance.json_data, best_solution))

Generation 1 - Best Makespan: 1066
Generation 2 - Best Makespan: 1049
Generation 3 - Best Makespan: 1049
Generation 4 - Best Makespan: 1049
Generation 5 - Best Makespan: 1049
Generation 6 - Best Makespan: 1049
Generation 7 - Best Makespan: 1049
Generation 8 - Best Makespan: 1049
Generation 9 - Best Makespan: 1049
Generation 10 - Best Makespan: 1049
Generation 11 - Best Makespan: 1049
Generation 12 - Best Makespan: 1049
Generation 13 - Best Makespan: 1049
Generation 14 - Best Makespan: 1049
Generation 15 - Best Makespan: 1049
Generation 16 - Best Makespan: 1049
Generation 17 - Best Makespan: 1049
Generation 18 - Best Makespan: 1049
Generation 19 - Best Makespan: 1049
Generation 20 - Best Makespan: 1049
Generation 21 - Best Makespan: 1049
Generation 22 - Best Makespan: 1049
Generation 23 - Best Makespan: 1049
Generation 24 - Best Makespan: 1049
Generation 25 - Best Makespan: 1049
Generation 26 - Best Makespan: 1049
Generation 27 - Best Makespan: 1049
Generation 28 - Best Makespan: 1049
G

In [ ]:
hard_instance = InstanceData('../examples/357_15_146_H.json')
evolutionary_algo_hard = GeneticAlgorithm(hard_instance)
best_solution_hard, progress_history_hard = evolutionary_algo_hard.evolutionary_algorithm(pop_size=50, nb_generation=50, mu=150, lamda=100)
print(f"Best solution found for hard instance: {best_solution_hard}")

Generation 1 - Best Makespan: 13722
Generation 2 - Best Makespan: 13722
Generation 3 - Best Makespan: 13388
Generation 4 - Best Makespan: 13388
Generation 5 - Best Makespan: 13388
Generation 6 - Best Makespan: 13388
Generation 7 - Best Makespan: 13388
Generation 8 - Best Makespan: 13388
Generation 9 - Best Makespan: 12768
Generation 10 - Best Makespan: 12768
Generation 11 - Best Makespan: 12768
Generation 12 - Best Makespan: 12768
Generation 13 - Best Makespan: 12705
Generation 14 - Best Makespan: 12705
Generation 15 - Best Makespan: 12705
Generation 16 - Best Makespan: 12705
Generation 17 - Best Makespan: 12705
Generation 18 - Best Makespan: 12705
Generation 19 - Best Makespan: 12705
Generation 20 - Best Makespan: 12705
Generation 21 - Best Makespan: 12705
Generation 22 - Best Makespan: 12264
Generation 23 - Best Makespan: 12264
Generation 24 - Best Makespan: 12264
Generation 25 - Best Makespan: 12264
Generation 26 - Best Makespan: 12264
Generation 27 - Best Makespan: 12264
Generation

AttributeError: 'dict' object has no attribute 'scheduling'